In [1]:
!pip install transformers torch pandas tqdm


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm


In [5]:
# Ganti dengan nama file CSV Anda
df = pd.read_csv(
    "yt_sahroni - Copy.csv",
    sep=";",
    engine="python"
)

# Pastikan kolom comment ada
print(df.columns)

# Ambil hanya kolom komentar
texts = df["comment"].astype(str).tolist()

print("Jumlah komentar:", len(texts))


Index(['comment', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9',
       'Unnamed: 10'],
      dtype='object')
Jumlah komentar: 17099


In [6]:
model_name = "indolem/indobertweet-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Gunakan GPU jika tersedia
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31923, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [7]:
def indobertweet_embedding(text_list, max_length=128):
    embeddings = []

    for text in tqdm(text_list):
        encoded = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            output = model(**encoded)

        # Mean Pooling
        token_embeddings = output.last_hidden_state
        attention_mask = encoded["attention_mask"]

        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        masked_embeddings = token_embeddings * mask

        summed = torch.sum(masked_embeddings, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)

        mean_pooled = summed / counts
        embeddings.append(mean_pooled.squeeze().cpu().numpy())

    return embeddings


In [8]:
embeddings = indobertweet_embedding(texts)

print("Jumlah embedding:", len(embeddings))
print("Dimensi embedding:", embeddings[0].shape)


100%|██████████| 17099/17099 [03:19<00:00, 85.59it/s] 

Jumlah embedding: 17099
Dimensi embedding: (768,)


In [9]:
embedding_df = pd.DataFrame(embeddings)
embedding_df.columns = [f"emb_{i}" for i in range(embedding_df.shape[1])]

# Gabungkan dengan data asli
final_df = pd.concat([df.reset_index(drop=True), embedding_df], axis=1)

# Simpan hasil embedding
final_df.to_csv("hasil_embedding_indobertweet.csv", index=False)

print("Embedding berhasil disimpan!")


Embedding berhasil disimpan!


In [10]:
!zip hasil_embedding_indobertweet.zip hasil_embedding_indobertweet.csv


  adding: hasil_embedding_indobertweet.csv (deflated 58%)


In [11]:
from google.colab import files

files.download("hasil_embedding_indobertweet.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import numpy as np

np.save("embedding_indobertweet.npy", embeddings)
from google.colab import files

files.download("embedding_indobertweet.npy")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
!zip embedding_indobertweet.zip embedding_indobertweet.npy


updating: embedding_indobertweet.npy (deflated 11%)
